# Módulo contexto de procesos
Agrega dos columnas a las tablas por tipo de proceso (capítulos 5 y 6):
`etapa_proceso_fuente` y `contexto_accion_penal`. Salen de mapas cerrados verificados
contra el PDF; un cuadro que no está inventariado hace fallar el ETL.

In [ ]:
import pandas as pd

ETAPA_AUDITADA_POR_CUADRO = {
    "5.3.1.1": "informes_inicio_investigacion",
    "5.3.1.2": "imputaciones_formales",
    "5.3.2.1": "causas",
    "6.3.1.1": "informes_inicio_investigacion",
    "6.3.1.2": "imputaciones_formales",
    "6.3.2.1": "causas",
}

CUADROS_NO_APLICA = {
    "5.1.1", "5.1.1.1", "5.1.1.2", "5.1.1.3", "5.1.1.5",
    "5.1.2.1", "5.1.2.2", "5.1.2.3", "5.1.2.4", "5.1.2.5",
    "5.1.2.6", "5.1.2.7", "5.1.3.1", "5.1.3.2", "5.1.3.3",
    "5.1.3.4", "5.1.3.5", "5.1.3.6", "5.1.3.7", "5.1.3.8",
    "5.1.3.9", "5.1.3.10", "5.1.3.11", "5.1.3.12",
    "5.2.1.1", "5.2.1.2", "5.2.1.3", "5.2.1.4", "5.2.1.5",
    "5.2.2.1", "5.2.2.2", "5.2.2.3", "5.2.2.4", "5.2.2.5",
    "5.3.1.3", "5.3.1.5", "5.3.2.2", "5.3.2.3", "5.3.2.4",
    "5.3.2.5", "5.3.2.6", "5.3.3.1", "5.3.3.2", "5.3.3.3",
    "5.3.3.4", "5.3.3.5", "5.3.4.1", "5.3.4.2", "5.3.4.3",
    "5.3.4.4",
    "6.1.1.1", "6.1.1.2", "6.1.1.3", "6.1.1.4", "6.1.1.5",
    "6.1.2.1", "6.1.2.2", "6.1.2.3", "6.1.2.4", "6.1.2.5",
    "6.1.2.6", "6.1.2.7", "6.1.3.1", "6.1.3.2", "6.1.3.3",
    "6.1.3.4", "6.1.3.5", "6.1.3.6", "6.1.3.7",
    "6.2.1.1", "6.2.1.2", "6.2.1.3", "6.2.1.4", "6.2.1.5",
    "6.3.1.3", "6.3.1.4", "6.3.1.5", "6.3.1.6",
    "6.3.2.2", "6.3.2.3", "6.3.2.4", "6.3.2.5", "6.3.2.6",
    "6.3.3.1", "6.3.3.2", "6.3.3.3", "6.3.3.4", "6.3.3.5",
    "6.3.3.6",
}

CUADROS_CONOCIDOS = set(ETAPA_AUDITADA_POR_CUADRO.keys()) | CUADROS_NO_APLICA

ETAPA_POR_CUADRO = {}
for cuadro in CUADROS_NO_APLICA:
    ETAPA_POR_CUADRO[cuadro] = "no_aplica"
for cuadro in ETAPA_AUDITADA_POR_CUADRO:
    ETAPA_POR_CUADRO[cuadro] = ETAPA_AUDITADA_POR_CUADRO[cuadro]

CONTEXTOS_ACCION_PENAL = ["penal_comun", "anticorrupcion", "violencia_mujeres"]

# En estos cuadros el contexto es el rótulo de la fila.
CUADROS_CONTEXTO_DIRECTO = {"5.3.1.1", "5.3.1.2", "6.3.1.1", "6.3.1.2"}
# En estos el contexto es una celda padre que abarca tres filas de acción penal (bloques de 3x3).
CUADROS_CONTEXTO_BLOQUES = {"5.3.2.1", "6.3.2.1"}
CUADROS_CONTEXTO_AUDITADO = CUADROS_CONTEXTO_DIRECTO | CUADROS_CONTEXTO_BLOQUES
CUADROS_CONTEXTO_NO_APLICA = CUADROS_CONOCIDOS - CUADROS_CONTEXTO_AUDITADO

CONTEXTO_DIRECTO_POR_ROTULO = {
    "PENAL COMUN": "penal_comun",
    "ANTICORRUPCIÓN": "anticorrupcion",
    "CONTRA LA VIOLENCIA HACIA LAS MUJERES": "violencia_mujeres",
}

CONTEXTOS_POR_CUADRO = {}
for cuadro in CUADROS_CONTEXTO_AUDITADO:
    CONTEXTOS_POR_CUADRO[cuadro] = CONTEXTOS_ACCION_PENAL

ESTRUCTURA_CONTEXTO_POR_CUADRO = {}
for cuadro in CUADROS_CONTEXTO_DIRECTO:
    ESTRUCTURA_CONTEXTO_POR_CUADRO[cuadro] = "rotulo_fila"
for cuadro in CUADROS_CONTEXTO_BLOQUES:
    ESTRUCTURA_CONTEXTO_POR_CUADRO[cuadro] = "bloque_padre_3x3"
for cuadro in CUADROS_CONTEXTO_NO_APLICA:
    ESTRUCTURA_CONTEXTO_POR_CUADRO[cuadro] = "no_aplica"

# Texto de la celda padre que tiene que aparecer dentro de cada bloque de tres filas.
MARCADORES_PADRE_POR_CUADRO = {
    "5.3.2.1": {
        "penal_comun": ["PENAL COMUN"],
        "anticorrupcion": ["ANTICORRUPCIÒN"],
        "violencia_mujeres": ["CONTRA LA VIOLENCIA HACIA LA"],
    },
    "6.3.2.1": {
        "penal_comun": ["PENAL COMUN"],
        "anticorrupcion": ["ANTICORRUPCIÒN"],
        "violencia_mujeres": ["CONTRA LA", "VIOLENCIA HACIA", "LAS MUJERES"],
    },
}
MARCADORES_ACCION = ["ACCIÓN PENAL PÙBLICA", "A INSTANCIA DE PARTE", "ACCIÓN PENAL PRIVADA"]

In [ ]:
def obtener_etapa_proceso(cuadro_origen):
    cuadro = str(cuadro_origen)
    if cuadro not in ETAPA_POR_CUADRO:
        raise ValueError("Cuadro desconocido para etapa: " + cuadro)
    return ETAPA_POR_CUADRO[cuadro]


def ordenar_filas(grupo):
    return grupo.sort_values(["pagina_pdf", "orden_fila"], kind="stable")


def validar_total_bloque(grupo, cuadro, entidad):
    totales = grupo[grupo["tipo_fila_derivado"] == "total"]
    if len(totales) != 1:
        raise ValueError(cuadro + "/" + str(entidad) + ": se esperaba una fila total; hay " + str(len(totales)))
    literal = str(totales.iloc[0]["tipo_proceso"])
    if not literal.startswith("TOTAL"):
        raise ValueError(cuadro + "/" + str(entidad) + ": rótulo de total inesperado: " + repr(literal))


def asignar_contexto_directo(resultado, cuadro, indices):
    subconjunto = resultado.loc[indices]
    for entidad, grupo in subconjunto.groupby("entidad", sort=False, dropna=False):
        grupo = ordenar_filas(grupo)
        detalle = grupo[grupo["tipo_fila_derivado"] == "detalle"]
        literales = detalle["tipo_proceso"].tolist()
        if literales != list(CONTEXTO_DIRECTO_POR_ROTULO.keys()):
            raise ValueError(cuadro + "/" + str(entidad) + ": secuencia penal inesperada: " + repr(literales))
        validar_total_bloque(grupo, cuadro, entidad)
        contextos = []
        for literal in literales:
            contextos.append(CONTEXTO_DIRECTO_POR_ROTULO[literal])
        resultado.loc[detalle.index, "contexto_accion_penal"] = contextos


def validar_acciones_bloque(detalle, cuadro, entidad):
    literales = detalle["tipo_proceso"].astype(str).tolist()
    if len(literales) != 9:
        raise ValueError(cuadro + "/" + str(entidad) + ": se esperaban 9 filas de acción; hay " + str(len(literales)))
    for i in range(3):
        contexto = CONTEXTOS_ACCION_PENAL[i]
        bloque = literales[i * 3:i * 3 + 3]
        for j in range(3):
            if MARCADORES_ACCION[j] not in bloque[j]:
                raise ValueError(cuadro + "/" + str(entidad) + "/" + contexto + ": acción inesperada " + repr(bloque[j]))
        texto_bloque = " ".join(bloque)
        for marcador in MARCADORES_PADRE_POR_CUADRO[cuadro][contexto]:
            if marcador not in texto_bloque:
                raise ValueError(cuadro + "/" + str(entidad) + "/" + contexto + ": falta el encabezado padre " + repr(marcador))


def asignar_contexto_bloques(resultado, cuadro, indices):
    subconjunto = resultado.loc[indices]
    for entidad, grupo in subconjunto.groupby("entidad", sort=False, dropna=False):
        grupo = ordenar_filas(grupo)
        detalle = grupo[grupo["tipo_fila_derivado"] == "detalle"]
        validar_acciones_bloque(detalle, cuadro, entidad)
        validar_total_bloque(grupo, cuadro, entidad)
        contextos = []
        for contexto in CONTEXTOS_ACCION_PENAL:
            contextos.append(contexto)
            contextos.append(contexto)
            contextos.append(contexto)
        resultado.loc[detalle.index, "contexto_accion_penal"] = contextos


def incorporar_contexto_procesos(df):
    requeridas = {"cuadro_origen", "pagina_pdf", "orden_fila", "entidad", "tipo_fila_derivado", "tipo_proceso"}
    faltantes = requeridas - set(df.columns)
    if len(faltantes) > 0:
        raise ValueError("Faltan columnas para derivar contexto: " + str(sorted(faltantes)))
    cuadros = set(df["cuadro_origen"].astype(str).unique())
    desconocidos = cuadros - CUADROS_CONOCIDOS
    if len(desconocidos) > 0:
        raise ValueError("Cuadros desconocidos para contexto: " + str(sorted(desconocidos)))

    resultado = df.copy()
    resultado["etapa_proceso_fuente"] = resultado["cuadro_origen"].map(obtener_etapa_proceso).astype("string")
    resultado["contexto_accion_penal"] = pd.Series("no_aplica", index=resultado.index, dtype="string")

    for cuadro in sorted(CUADROS_CONTEXTO_DIRECTO & cuadros):
        indices = resultado.index[resultado["cuadro_origen"] == cuadro]
        asignar_contexto_directo(resultado, cuadro, indices)
    for cuadro in sorted(CUADROS_CONTEXTO_BLOQUES & cuadros):
        indices = resultado.index[resultado["cuadro_origen"] == cuadro]
        asignar_contexto_bloques(resultado, cuadro, indices)

    if resultado[["etapa_proceso_fuente", "contexto_accion_penal"]].isna().any().any():
        raise ValueError("La derivación produjo contextos nulos")
    return resultado